# Set Library Path

In [1]:
.libPaths("/home/vhaghani/anaconda3/envs/jupyter_nb_R4.4.2/lib/R/library")

# Load Libraries

In [3]:
library(readxl)
library(ggplot2)
library(viridis)
library(dplyr)
library(glue)
library(tidyr)

# Set Modules and GO Databases

In [4]:
# Subset to significant modules only
significant_associations <- read.table("08_WGCNA/MECP2_significant_associations.txt", header = TRUE, sep = "\t", stringsAsFactors = FALSE)

# Remove ME prefix from module name
significant_associations$module <- sub("^ME", "", significant_associations$module)

# Remove grey module
significant_associations <- significant_associations[significant_associations$module != "grey", ]

# Remove insignificant modules
significant_associations <- significant_associations[significant_associations$pvalue < 0.05, ]

# Create list of significant modules
modules <- unique(significant_associations$module)

# View
print(modules)

 [1] "coral2"          "darkorange"      "lightyellow"     "yellowgreen"    
 [5] "lightgreen"      "plum"            "mediumorchid"    "saddlebrown"    
 [9] "violet"          "lightcyan1"      "darkred"         "turquoise"      
[13] "thistle1"        "lavenderblush3"  "orangered4"      "brown4"         
[17] "darkgreen"       "green"           "salmon"          "grey60"         
[21] "orange"          "orangered3"      "darkslateblue"   "skyblue"        
[25] "floralwhite"     "royalblue"       "darkorange2"     "brown"          
[29] "honeydew1"       "blue"            "antiquewhite4"   "steelblue"      
[33] "purple"          "darkseagreen4"   "paleturquoise"   "lightsteelblue1"
[37] "lightcyan"       "red"             "navajowhite2"    "yellow"         
[41] "thistle2"        "mediumpurple3"   "greenyellow"     "salmon4"        
[45] "pink"            "skyblue2"        "darkturquoise"   "ivory"          
[49] "coral1"          "bisque4"         "midnightblue"    "palevioletred3" 

In [5]:
# List of GO databases
databases <- c("GO_Biological_Process_2025", "GO_Cellular_Component_2025", "GO_Molecular_Function_2025",
               "KEGG_2021_Human", "Panther_2016", "RNAseq_DiseaseGene_DrugSigs_GEO", "Reactome_2022")

# Visualize GO Results

In [6]:
# Store GO data into data frame

# Initialize an empty dataframe
all_data <- data.frame()

# Iterate over each module and read in the corresponding Excel file
for (module in modules) {
  file_path <- glue("09_WGCNA_GO/MECP2/{module}_enrichr_results.xlsx")
  
  for (database in databases) {
    try({
      # Read the data from the Excel file
      df <- read_excel(file_path, sheet = database)
      
      if (nrow(df) == 0) {
        next  # Skip to the next database if no data
      }
      
      # Select the required columns and add module and database information
      df <- df %>%
        select(Term, Adjusted.P.value, Odds.Ratio) %>%
        filter(Adjusted.P.value <= 0.05) %>%
        mutate(Module = module, Database = database)
      
      # Append to the dataframe
      all_data <- bind_rows(all_data, df)
      
    }, silent = TRUE)
  }
}

# Check if any data was read
if (nrow(all_data) == 0) {
  stop("No data read from any of the Excel files.")
}

# Display the combined dataframe
head(all_data)

,Term,Adjusted.P.value,Odds.Ratio,Module,Database
,<chr>,<dbl>,<dbl>,<chr>,<chr>
1,Embryonic Skeletal System Morphogenesis (GO:0048704),0.0128237286,19.819090,lightgreen,GO_Biological_Process_2025
2,Skeletal System Morphogenesis (GO:0048705),0.0183027362,14.729840,lightgreen,GO_Biological_Process_2025
3,Myelination in Peripheral Nervous System (GO:0022011),0.0426360226,110.844444,thistle1,GO_Biological_Process_2025
4,BZLT Akata Knockdown GSE79304 down,0.0000456522,3.088054,green,RNAseq_DiseaseGene_DrugSigs_GEO
5,ELAVL2 NPC Knockdown GSE69092 down,0.0000456522,3.000065,green,RNAseq_DiseaseGene_DrugSigs_GEO
6,RBFOX2 HEK293T knockdown GSE60391 up,0.0000456522,2.993414,green,RNAseq_DiseaseGene_DrugSigs_GEO


In [7]:
# Count the number of total terms and unique terms for each database
database_term_counts <- all_data %>%
  group_by(Database) %>%
  summarise(
    Total_Terms = n(),
    Unique_Terms = n_distinct(Term)
  )

# Print the table
print(database_term_counts)

# A tibble: 6 × 3
  Database                        Total_Terms Unique_Terms
  <chr>                                 <int>        <int>
1 GO_Biological_Process_2025               20           20
2 GO_Cellular_Component_2025                3            3
3 GO_Molecular_Function_2025                6            6
4 Panther_2016                              3            3
5 RNAseq_DiseaseGene_DrugSigs_GEO         362          238
6 Reactome_2022                             5            5


In [8]:
# Calculate the number of modules each term appears in for each database
term_module_counts <- all_data %>%
  group_by(Database, Term) %>%
  summarise(ModuleCount = n_distinct(Module), .groups = 'drop')

# Rank the terms within each database by the number of modules they appear in
ranked_terms <- term_module_counts %>%
  arrange(Database, desc(ModuleCount)) %>%
  group_by(Database) %>%
  slice_head(n = 50) %>%
  ungroup()

# Merge with the original data to filter the top 25 terms per database
filtered_data_top_25 <- all_data %>%
  semi_join(ranked_terms, by = c("Database", "Term"))

# Print the filtered data
head(filtered_data_top_25)

,Term,Adjusted.P.value,Odds.Ratio,Module,Database
,<chr>,<dbl>,<dbl>,<chr>,<chr>
1,Embryonic Skeletal System Morphogenesis (GO:0048704),0.0128237286,19.819090,lightgreen,GO_Biological_Process_2025
2,Skeletal System Morphogenesis (GO:0048705),0.0183027362,14.729840,lightgreen,GO_Biological_Process_2025
3,Myelination in Peripheral Nervous System (GO:0022011),0.0426360226,110.844444,thistle1,GO_Biological_Process_2025
4,BZLT Akata Knockdown GSE79304 down,0.0000456522,3.088054,green,RNAseq_DiseaseGene_DrugSigs_GEO
5,ELAVL2 NPC Knockdown GSE69092 down,0.0000456522,3.000065,green,RNAseq_DiseaseGene_DrugSigs_GEO
6,RBFOX2 HEK293T knockdown GSE60391 up,0.0000456522,2.993414,green,RNAseq_DiseaseGene_DrugSigs_GEO


In [9]:
# Count the number of total terms and unique terms for each database
filtered_database_term_counts <- filtered_data_top_25 %>%
  group_by(Database) %>%
  summarise(
    Total_Terms = n(),
    Unique_Terms = n_distinct(Term)
  )

# Print the table
print(filtered_database_term_counts)

# A tibble: 6 × 3
  Database                        Total_Terms Unique_Terms
  <chr>                                 <int>        <int>
1 GO_Biological_Process_2025               20           20
2 GO_Cellular_Component_2025                3            3
3 GO_Molecular_Function_2025                6            6
4 Panther_2016                              3            3
5 RNAseq_DiseaseGene_DrugSigs_GEO         124           50
6 Reactome_2022                             5            5


In [10]:
# Create plots for each database
for (database in unique(filtered_data_top_25$Database)) {
  database_filtered_data <- filtered_data_top_25 %>%
    filter(Database == database)
  
  if (nrow(database_filtered_data) > 0) {
    
    # Calculate dynamic height based on number of unique terms
    n_terms <- length(unique(database_filtered_data$Term))
    plot_height <- ifelse(n_terms > 25, 11, 7)
    
    dot_plot <- ggplot(database_filtered_data, aes(x = Module, y = Term, size = Odds.Ratio, fill = Adjusted.P.value)) +
      geom_point(shape = 21) +
      scale_fill_viridis() +
      xlab('') + ylab('') +
      labs(
        title = 'Top Enrichr Terms Across Modules',
        subtitle = glue('{database} (n_terms = {n_terms})')
      ) +
      theme(
        panel.background = element_rect(fill = "white", color = NA),
        panel.grid.major = element_blank(),
        panel.grid.minor = element_blank(),
        panel.border = element_blank(),
        axis.line = element_line(color = "black"),
        plot.background = element_rect(fill = "white", color = NA),
        axis.text.x = element_text(angle = 90, hjust = 1)
      )
    
    # Save the dot plot with the database name in the filename
    ggsave(filename = glue("09_WGCNA_GO/top_50_dot_plot_{database}.pdf"), 
           plot = dot_plot, 
           height = plot_height, 
           width = 15)
    
    cat(glue("Saved {database} plot with height {plot_height} (n_terms = {n_terms})\n"))
    
  } else {
    cat(glue("No data available for {database}. Skipping...\n"))
  }
}

Saved GO_Biological_Process_2025 plot with height 7 (n_terms = 20)Saved RNAseq_DiseaseGene_DrugSigs_GEO plot with height 11 (n_terms = 50)Saved Panther_2016 plot with height 7 (n_terms = 3)Saved GO_Molecular_Function_2025 plot with height 7 (n_terms = 6)Saved Reactome_2022 plot with height 7 (n_terms = 5)Saved GO_Cellular_Component_2025 plot with height 7 (n_terms = 3)